# Project 2: Student Marks Prediction

**Objective:** build a Machine Learning model that predicts a student's **final exam marks** from academic and study-related factors, using **Linear Regression**.

**Features:** study hours, attendance %, previous exam marks, assignment marks, internal marks, practice test score, sleep hours (+ a few categorical background columns).
**Target:** `final_exam_marks` (0–100).

**Steps covered in this notebook**
1. Data Collection
2. Data Preprocessing (missing values, duplicates, categorical variables, outliers, feature selection)
3. Model Building (train/test split, Linear Regression, predictions)
4. Model Evaluation (MAE, MSE, RMSE, R²)
5. Save Model (joblib)
6. Streamlit UI
7. Interpretation

Run the cells from top to bottom (`Runtime → Run all` works too).

## 0. Setup

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q streamlit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Running in Colab:", IN_COLAB)

## 1. Data Collection

The dataset is `student_marks.csv` (1,020 rows). If you uploaded it to Colab (Files panel on the left), it is loaded directly.
If it is not found, the cell below regenerates exactly the same data with the code from `generate_dataset.py` (fixed random seed), so the notebook always runs.

In [ ]:
import numpy as np
import pandas as pd


def generate_student_data(n=1000, seed=42):
    rng = np.random.default_rng(seed)

    # Hidden "ability" drives the correlated academic scores
    ability = rng.normal(0, 1, n)

    gender = rng.choice(["Male", "Female"], n)
    parental_education = rng.choice(
        ["High School", "Bachelor", "Master", "PhD"], n, p=[0.35, 0.40, 0.18, 0.07]
    )
    internet_access = rng.choice(["Yes", "No"], n, p=[0.82, 0.18])
    extracurricular = rng.choice(["Yes", "No"], n, p=[0.45, 0.55])

    study_hours = np.clip(rng.normal(5 + 0.8 * ability, 2.0), 0.5, 12)
    attendance_pct = np.clip(rng.normal(80 + 5 * ability, 9), 40, 100)
    previous_exam_marks = np.clip(rng.normal(65 + 10 * ability, 7), 20, 100)
    assignment_marks = np.clip(rng.normal(70 + 6 * ability, 10), 20, 100)
    internal_marks = np.clip(rng.normal(64 + 8 * ability, 8), 15, 100)
    practice_test_score = np.clip(rng.normal(62 + 9 * ability, 9), 10, 100)
    sleep_hours = np.clip(rng.normal(7, 1.2, n), 3, 10)
    commute_minutes = np.clip(rng.normal(30, 15, n), 2, 90)  # irrelevant feature

    edu_effect = pd.Series(parental_education).map(
        {"High School": 0.0, "Bachelor": 1.0, "Master": 2.0, "PhD": 3.0}
    ).to_numpy()

    final_exam_marks = (
        -8
        + 0.25 * previous_exam_marks
        + 0.15 * internal_marks
        + 0.08 * assignment_marks
        + 0.15 * practice_test_score
        + 2.0 * study_hours
        + 0.15 * attendance_pct
        + 1.0 * sleep_hours
        + edu_effect
        + np.where(internet_access == "Yes", 2.0, 0.0)
        + np.where(extracurricular == "Yes", 1.0, 0.0)
        + rng.normal(0, 4, n)
    )
    final_exam_marks = np.clip(final_exam_marks, 0, 100)

    df = pd.DataFrame({
        "student_id": [f"S{i:04d}" for i in range(1, n + 1)],
        "gender": gender,
        "parental_education": parental_education,
        "internet_access": internet_access,
        "extracurricular": extracurricular,
        "study_hours": study_hours.round(1),
        "attendance_pct": attendance_pct.round(1),
        "previous_exam_marks": previous_exam_marks.round(0),
        "assignment_marks": assignment_marks.round(0),
        "internal_marks": internal_marks.round(0),
        "practice_test_score": practice_test_score.round(0),
        "sleep_hours": sleep_hours.round(1),
        "commute_minutes": commute_minutes.round(0),
        "final_exam_marks": final_exam_marks.round(1),
    })

    # Outliers / data-entry errors
    idx = rng.choice(n, 11, replace=False)
    df.loc[idx[:5], "study_hours"] = rng.uniform(25, 40, 5).round(1)
    df.loc[idx[5:8], "attendance_pct"] = rng.uniform(120, 150, 3).round(1)
    df.loc[idx[8:], "sleep_hours"] = rng.uniform(18, 20, 3).round(1)

    # Missing values (~3% in selected columns)
    for col in ["study_hours", "attendance_pct", "sleep_hours",
                "assignment_marks", "parental_education", "internet_access"]:
        miss = rng.choice(n, int(0.03 * n), replace=False)
        df.loc[miss, col] = np.nan

    # Duplicate rows
    dups = df.sample(20, random_state=seed)
    df = pd.concat([df, dups], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    return df


CSV_PATH = "student_marks.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Loaded", CSV_PATH)
else:
    df = generate_student_data()
    df.to_csv(CSV_PATH, index=False)
    print("CSV not found - generated and saved", CSV_PATH)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

### Understanding each column

| Column | Type | Meaning |
|---|---|---|
| `student_id` | ID | Unique student identifier (not a feature) |
| `gender` | categorical | Male / Female |
| `parental_education` | categorical | Highest education of parents: High School, Bachelor, Master, PhD |
| `internet_access` | categorical | Internet access at home: Yes / No |
| `extracurricular` | categorical | Takes part in extracurricular activities: Yes / No |
| `study_hours` | numeric | Average self-study hours per day |
| `attendance_pct` | numeric | Class attendance percentage (0–100) |
| `previous_exam_marks` | numeric | Marks in the previous exam (out of 100) |
| `assignment_marks` | numeric | Average assignment marks (out of 100) |
| `internal_marks` | numeric | Internal assessment marks (out of 100) |
| `practice_test_score` | numeric | Average practice test score (out of 100) |
| `sleep_hours` | numeric | Average sleep hours per day |
| `commute_minutes` | numeric | Daily travel time to college in minutes |
| `final_exam_marks` | numeric | **Target** – final exam marks (out of 100) |

## 2. Data Preprocessing

### 2.1 Missing values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "percent": missing_pct})[missing > 0]

Numeric columns are filled with the **median** (not affected by outliers); categorical columns with the **mode** (most frequent value).

In [ ]:
TARGET = "final_exam_marks"
num_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(exclude="number").columns.drop("student_id").tolist()

fill_values = {}
for col in num_cols:
    fill_values[col] = df[col].median()
for col in cat_cols:
    fill_values[col] = df[col].mode()[0]

df = df.fillna(fill_values)
print("Missing values left:", df.isnull().sum().sum())
print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

### 2.2 Duplicates

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df = df.drop(columns=["student_id"])   # identifier, carries no information
print("Shape after removing duplicates:", df.shape)

### 2.3 Outliers

Boxplots and the IQR rule (values outside `Q1 - 1.5·IQR` and `Q3 + 1.5·IQR`) show which columns contain outliers.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(y=df[col], ax=ax, color="skyblue")
    ax.set_title(col)
plt.tight_layout()
plt.show()

def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

rows = []
for col in num_cols:
    lo, hi = iqr_bounds(df[col])
    rows.append({"column": col, "lower": round(lo, 2), "upper": round(hi, 2),
                 "outliers": int(((df[col] < lo) | (df[col] > hi)).sum())})
pd.DataFrame(rows)

Two kinds of outliers show up:
- **Impossible values** (data-entry errors): study hours of 25–40 per day, attendance above 100 %, 18+ hours of sleep. These are replaced with the column median.
- **Genuine extreme values**, e.g. a very strong student scoring 98 or a weak one scoring 30. These are real students, so they are **kept**; removing them would teach the model that such students do not exist.

In [ ]:
valid_range = {
    "study_hours": (0, 16), "attendance_pct": (0, 100), "sleep_hours": (0, 12),
    "previous_exam_marks": (0, 100), "assignment_marks": (0, 100),
    "internal_marks": (0, 100), "practice_test_score": (0, 100), "commute_minutes": (0, 180),
}
for col in num_cols:
    vlo, vhi = valid_range[col]
    invalid = (df[col] < vlo) | (df[col] > vhi)
    if invalid.any():
        print(f"{col}: {invalid.sum()} impossible values replaced with median {df[col].median()}")
    df.loc[invalid, col] = df[col].median()

df[num_cols].describe().T[["min", "max", "mean"]]

### 2.4 Exploratory look at the data

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.scatterplot(x=df[col], y=df[TARGET], ax=ax, s=12, alpha=0.6)
    ax.set_title(f"{col} vs marks")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    sns.boxplot(x=df[col], y=df[TARGET], ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### 2.5 Handling categorical variables

Linear Regression needs numbers, so categorical columns are **one-hot encoded** with `pd.get_dummies(drop_first=True)`.
`drop_first=True` removes one level per column to avoid perfect multicollinearity (the dummy variable trap).

In [ ]:
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df_encoded.head()

### 2.6 Feature selection

We look at the correlation of every feature with the target and drop features whose absolute correlation is below **0.10** (they add noise, not information).

In [ ]:
corr = df_encoded.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Correlation matrix")
plt.show()

target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
plt.figure(figsize=(8, 5))
target_corr.plot(kind="barh", color=["tab:green" if v > 0 else "tab:red" for v in target_corr])
plt.gca().invert_yaxis()
plt.title("Correlation with final_exam_marks")
plt.show()
target_corr.round(3)

In [ ]:
THRESHOLD = 0.10
selected_features = target_corr[target_corr.abs() >= THRESHOLD].index.tolist()
dropped_features = target_corr[target_corr.abs() < THRESHOLD].index.tolist()
print("Selected features:", selected_features)
print("Dropped features :", dropped_features)

## 3. Model Building

### 3.1 Train / test split (80 / 20)

In [ ]:
X = df_encoded[selected_features]
y = df_encoded[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

### 3.2 Feature scaling

`StandardScaler` puts all features on the same scale (mean 0, std 1). It is **fit on the training data only** to avoid leaking information from the test set.
Scaling also makes the regression coefficients directly comparable in the interpretation step.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 3.3 Train Linear Regression and generate predictions

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

print("Intercept:", round(model.intercept_, 3))
pd.DataFrame({"actual": y_test.values[:10], "predicted": y_test_pred[:10].round(1)})

## 4. Model Evaluation

In [ ]:
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": np.sqrt(mse), "R2": r2_score(y_true, y_pred)}

results = pd.DataFrame({"Train": evaluate(y_train, y_train_pred),
                        "Test": evaluate(y_test, y_test_pred)}).round(3)
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_test_pred, alpha=0.6)
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
axes[0].plot(lims, lims, "r--", label="perfect prediction")
axes[0].set_xlabel("Actual marks"); axes[0].set_ylabel("Predicted marks")
axes[0].set_title("Actual vs Predicted (test set)"); axes[0].legend()

residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals, alpha=0.6)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted marks"); axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Residual plot")
plt.tight_layout()
plt.show()

sns.histplot(residuals, kde=True)
plt.title("Distribution of residuals")
plt.show()

## 5. Save Model

Saved into the `model/` folder:
- `model.pkl` – trained Linear Regression model
- `scaler.pkl` – fitted StandardScaler
- `artifacts.pkl` – feature column order, input ranges for the UI, fill values and metrics (everything the app needs to preprocess new input the same way)

In [ ]:
os.makedirs("model", exist_ok=True)

# raw input columns the UI must ask for (only those that survived feature selection)
numeric_inputs = [c for c in num_cols if c in selected_features]
categorical_inputs = [c for c in cat_cols if any(f.startswith(c + "_") for f in selected_features)]

numeric_features = {}
for c in numeric_inputs:
    step = 1.0 if c in ["previous_exam_marks", "assignment_marks", "internal_marks",
                        "practice_test_score", "commute_minutes"] else 0.5
    numeric_features[c] = {"min": float(np.floor(df[c].min())), "max": float(np.ceil(df[c].max())),
                           "default": float(round(df[c].median() / step) * step), "step": step}

categorical_features = {c: sorted(df[c].unique().tolist()) for c in categorical_inputs}

artifacts = {
    "feature_columns": selected_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "fill_values": fill_values,
    "metrics": {"test_mae": results.loc["MAE", "Test"], "test_mse": results.loc["MSE", "Test"],
                "test_rmse": results.loc["RMSE", "Test"], "test_r2": results.loc["R2", "Test"]},
}

joblib.dump(model, "model/model.pkl")
joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(artifacts, "model/artifacts.pkl")
print("Saved:", os.listdir("model"))

Quick check: reload the saved files and predict for one new student, using the same preprocessing the app uses.

In [ ]:
loaded_model = joblib.load("model/model.pkl")
loaded_scaler = joblib.load("model/scaler.pkl")
loaded_art = joblib.load("model/artifacts.pkl")

new_student = {c: cfg["default"] for c, cfg in loaded_art["numeric_features"].items()}
new_student.update({c: opts[0] for c, opts in loaded_art["categorical_features"].items()})
new_student["study_hours"] = 8
new_student["previous_exam_marks"] = 80

x = pd.get_dummies(pd.DataFrame([new_student]), columns=list(loaded_art["categorical_features"]))
x = x.reindex(columns=loaded_art["feature_columns"], fill_value=0).astype(float)
print(new_student)
print("Predicted final marks:", round(float(loaded_model.predict(loaded_scaler.transform(x))[0]), 1))

## 6. Streamlit UI

The next cell writes `app.py` (the same file that ships with the project). The app lets the user enter student details and shows the **predicted final marks** and an **input summary**.

In [ ]:
%%writefile app.py
"""
Streamlit UI for the Student Marks Prediction project.
Run:  streamlit run app.py
Needs the files created by the notebook in the model/ folder.
"""
import os

import joblib
import numpy as np
import pandas as pd
import streamlit as st

MODEL_DIR = "model"

st.set_page_config(page_title="Student Marks Predictor", layout="centered")


@st.cache_resource
def load_artifacts():
    model = joblib.load(os.path.join(MODEL_DIR, "model.pkl"))
    scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
    artifacts = joblib.load(os.path.join(MODEL_DIR, "artifacts.pkl"))
    return model, scaler, artifacts


def grade_for(marks):
    if marks >= 90:
        return "A+"
    if marks >= 80:
        return "A"
    if marks >= 70:
        return "B"
    if marks >= 60:
        return "C"
    if marks >= 50:
        return "D"
    if marks >= 40:
        return "E (Pass)"
    return "F (Fail)"


def prepare_input(raw, artifacts):
    """Apply the same encoding, column order and scaling used in training."""
    df = pd.DataFrame([raw])
    df = pd.get_dummies(df, columns=list(artifacts["categorical_features"].keys()))
    df = df.reindex(columns=artifacts["feature_columns"], fill_value=0)
    return df.astype(float)


LABELS = {
    "study_hours": "Study hours per day",
    "attendance_pct": "Attendance (%)",
    "previous_exam_marks": "Previous exam marks (out of 100)",
    "assignment_marks": "Assignment marks (out of 100)",
    "internal_marks": "Internal marks (out of 100)",
    "practice_test_score": "Practice test score (out of 100)",
    "sleep_hours": "Sleep hours per day",
    "commute_minutes": "Commute time (minutes)",
    "gender": "Gender",
    "parental_education": "Parental education",
    "internet_access": "Internet access at home",
    "extracurricular": "Extracurricular activities",
}

st.title("Student Final Marks Predictor")
st.write("Enter the student's details to predict the final exam marks "
         "(Linear Regression model).")

if not os.path.exists(os.path.join(MODEL_DIR, "model.pkl")):
    st.error("model/model.pkl not found. Run the notebook first to train and save the model.")
    st.stop()

model, scaler, artifacts = load_artifacts()

with st.form("student_form"):
    raw = {}
    col1, col2 = st.columns(2)
    items = list(artifacts["numeric_features"].items())
    for i, (col, cfg) in enumerate(items):
        target_col = col1 if i % 2 == 0 else col2
        raw[col] = target_col.slider(
            LABELS.get(col, col),
            min_value=float(cfg["min"]),
            max_value=float(cfg["max"]),
            value=float(cfg["default"]),
            step=float(cfg["step"]),
        )
    for col, options in artifacts["categorical_features"].items():
        raw[col] = st.selectbox(LABELS.get(col, col), options)
    submitted = st.form_submit_button("Predict final marks")

if submitted:
    X = prepare_input(raw, artifacts)
    X_scaled = scaler.transform(X)
    pred = float(np.clip(model.predict(X_scaled)[0], 0, 100))

    st.subheader("Prediction")
    c1, c2 = st.columns(2)
    c1.metric("Predicted final marks", f"{pred:.1f} / 100")
    c2.metric("Expected grade", grade_for(pred))

    rmse = artifacts.get("metrics", {}).get("test_rmse")
    if rmse is not None:
        st.caption(f"Typical error of the model on unseen data: about ±{rmse:.1f} marks (test RMSE).")

    st.subheader("Input summary")
    summary = pd.DataFrame(
        {"Feature": [LABELS.get(k, k) for k in raw], "Value": [str(v) for v in raw.values()]}
    )
    st.table(summary)

with st.expander("About the model"):
    m = artifacts.get("metrics", {})
    if m:
        st.write(f"Test R²: **{m['test_r2']:.3f}**, MAE: {m['test_mae']:.2f}, "
                 f"RMSE: {m['test_rmse']:.2f}")
    st.write("Features used:", ", ".join(artifacts["feature_columns"]))

### Launch the app from Colab

Colab cannot open `localhost` directly, so the cell below starts Streamlit and opens a free **Cloudflare tunnel**.
Wait until it prints a `https://....trycloudflare.com` link, then click it. The app keeps running while this Colab session is alive.
(When run outside Colab, the cell only prints the local command.)

In [ ]:
import subprocess, time, re

if IN_COLAB:
    if not os.path.exists("cloudflared"):
        !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x cloudflared

    streamlit_proc = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
        stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)

    tunnel_proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
        stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)

    url = None
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cloudflared.log").read())
        if m:
            url = m.group(0)
            break
    print("Open the app here:", url if url else "tunnel not ready - check cloudflared.log")
else:
    print("Run locally with:  streamlit run app.py")

If the Cloudflare link does not work, an alternative is localtunnel:
```
!npm install -g localtunnel
!curl -s ipv4.icanhazip.com      # this IP is the tunnel password
!npx localtunnel --port 8501
```

## 7. Interpretation

### 7.1 Which factors affect student marks?

In [ ]:
coef = pd.Series(model.coef_, index=selected_features).sort_values(key=abs, ascending=False)
plt.figure(figsize=(8, 5))
coef.plot(kind="barh", color=["tab:green" if v > 0 else "tab:red" for v in coef])
plt.gca().invert_yaxis()
plt.title("Standardized Linear Regression coefficients\n(change in marks for +1 std of the feature)")
plt.xlabel("Coefficient")
plt.show()
coef.round(3).to_frame("coefficient")

In [ ]:
m = artifacts["metrics"]
print(f"Test R2   : {m['test_r2']:.3f}  -> the model explains {m['test_r2']*100:.1f}% of the variation in final marks")
print(f"Test MAE  : {m['test_mae']:.2f} marks  -> on average a prediction is off by this many marks")
print(f"Test RMSE : {m['test_rmse']:.2f} marks  -> typical error, penalising large mistakes more")
print(f"Top 3 factors: {', '.join(coef.index[:3])}")

**Which factors affect student marks?**
Because features were standardized, the size of each coefficient shows its importance. **Study hours** and **previous exam marks** have the largest positive effect, followed by **practice test score**, **internal marks**, **attendance** and **assignment marks**. **Sleep hours** and **internet access** have smaller positive effects.
`commute_minutes` and `gender` had essentially zero correlation with marks, so feature selection removed them. Parental education and extracurricular activities have a small real effect, but it was too weak to pass the 0.10 correlation threshold and they were dropped as well.
Note that the academic scores are correlated with each other (a strong student scores well on all of them), so their individual coefficients share credit and should not be read as purely independent causes.

**What does the R² score mean?**
R² is the fraction of the variance in final marks that the model explains. R² = 1 would be perfect, R² = 0 means the model is no better than always predicting the average mark. The printed test R² (around 0.8) means roughly 80 % of the differences between students' marks are explained by the input features; the rest comes from factors not in the data (and random noise). Train and test R² are close, so the model is not overfitting.

**How accurate is the model?**
The test MAE is about 3–4 marks and RMSE about 4–5 marks on a 0–100 scale: a typical prediction is within ±5 marks of the real result. That is good enough to spot students at risk or estimate a likely grade band, but not to predict an exact score.

**What are the limitations?**
- The dataset is synthetic; relationships in real college data will be messier and the model must be retrained on real records before use.
- Linear Regression assumes a straight-line relationship. Effects that level off (e.g. sleep beyond 8 hours, study beyond 10 hours) are not captured.
- Correlated inputs (multicollinearity) make individual coefficients unstable, even though predictions stay good.
- Important factors such as motivation, teaching quality, health or exam difficulty are not in the data.
- Predictions are only reliable inside the range of the training data; very unusual inputs can give unrealistic marks (the app clips output to 0–100).
- Correlation is not causation: the model shows association, e.g. more study hours are linked to higher marks, but it does not prove that forcing extra hours will raise marks by the coefficient amount.

## (Optional) Download the trained model and app

In [ ]:
if IN_COLAB:
    !zip -q -r student_marks_project.zip model app.py student_marks.csv
    from google.colab import files
    files.download("student_marks_project.zip")
else:
    print("Files are already in the project folder.")